# 003 · Time Zones with zoneinfo

> **Standard Library › Dates & Times** · Python 3.14

A `datetime` can be *naive* (no time zone) or *aware* (it carries a `tzinfo`). The `zoneinfo` module (Python 3.9+) supplies real IANA time zones such as `Europe/London`, with full daylight-saving rules — the job `TimeZoneInfo` does in .NET.

**In this notebook you will learn:**

- naive vs aware datetimes, and how they relate to `DateTimeKind` and `DateTimeOffset`
- fixed offsets with `timezone.utc` / `timezone(timedelta(...))`
- named zones with `ZoneInfo("Europe/London")` and converting with `astimezone`
- what happens at DST transitions: ambiguous times, `fold`, and non-existent times
- why `datetime.utcnow()` is deprecated, and the "store UTC, convert at the edges" rule

## .NET mental model

| .NET | Python |
|---|---|
| `DateTime` with `Kind = Unspecified` | naive `datetime` (`tzinfo is None`) |
| `DateTime` with `Kind = Local` | naive `datetime` you *treat* as local — Python has no "local" flag |
| `DateTime` with `Kind = Utc` | aware `datetime` with `tzinfo=timezone.utc` |
| `DateTimeOffset` | aware `datetime` with a fixed-offset `timezone(...)` |
| `TimeZoneInfo.FindSystemTimeZoneById("Europe/London")` | `ZoneInfo("Europe/London")` |
| `TimeZoneInfo.ConvertTime(dto, tz)` | `dt.astimezone(tz)` |
| `TimeZoneInfo.IsAmbiguousTime` / `IsInvalidTime` | `fold` attribute + a UTC round-trip check |
| `DateTime.UtcNow` / `DateTimeOffset.UtcNow` | `datetime.now(timezone.utc)` |

A Python `datetime` with a `ZoneInfo` attached is closer to NodaTime's `ZonedDateTime` than to anything in the BCL: it remembers the *zone* (with its rules), not just the current offset.

## Naive vs aware

A **naive** datetime is just a calendar date and a clock reading — it could be in any zone. An **aware** datetime has a `tzinfo` and therefore represents an exact instant.

```csharp
// C#
var unspecified = new DateTime(2026, 7, 1, 9, 0, 0);                    // Kind = Unspecified
var utc         = new DateTime(2026, 7, 1, 9, 0, 0, DateTimeKind.Utc);
var withOffset  = new DateTimeOffset(2026, 7, 1, 9, 0, 0, TimeSpan.FromHours(1));
```

In [ ]:
from datetime import datetime, timedelta, timezone, UTC

naive = datetime(2026, 7, 1, 9, 0)
utc = datetime(2026, 7, 1, 9, 0, tzinfo=timezone.utc)
plus_one = datetime(2026, 7, 1, 9, 0, tzinfo=timezone(timedelta(hours=1)))

for value in (naive, utc, plus_one):
    print(f"{value!s:28} tzinfo={value.tzinfo!s:10} utcoffset={value.utcoffset()}")

`datetime.UTC` (Python 3.11+) is simply an alias for `timezone.utc`. `timezone(...)` creates a **fixed offset** — it knows nothing about daylight saving, exactly like the offset in a `DateTimeOffset`.

In [ ]:
print(UTC is timezone.utc)                                 # True
print(timezone(timedelta(hours=5, minutes=30)))            # UTC+05:30
print(timezone(timedelta(hours=-5), "EST"))                # a named fixed offset

⚠️ **Gotcha:** naive and aware datetimes cannot be ordered against each other — Python refuses to guess. Equality does not raise, it is simply always `False`:

In [ ]:
try:
    naive < utc
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(naive == utc)        # False — a naive and an aware value are never equal

## Named zones with `zoneinfo`

`ZoneInfo(key)` loads a zone from the IANA time zone database. Attaching it to a datetime gives you the correct offset **for that date**, including daylight saving time.

```csharp
// C# (.NET 6+ accepts IANA ids on every OS)
var london = TimeZoneInfo.FindSystemTimeZoneById("Europe/London");
var offset = london.GetUtcOffset(new DateTime(2026, 7, 1, 9, 0, 0));   // 01:00:00
```

In [ ]:
from zoneinfo import ZoneInfo

london = ZoneInfo("Europe/London")

winter = datetime(2026, 1, 15, 9, 0, tzinfo=london)
summer = datetime(2026, 7, 1, 9, 0, tzinfo=london)

for value in (winter, summer):
    print(value, value.tzname(), "offset:", value.utcoffset(), "dst:", value.dst())

`ZoneInfo` objects are cached, so creating the same key twice returns the same object. Unknown keys raise `ZoneInfoNotFoundError` (a subclass of `KeyError`; C#: `TimeZoneNotFoundException`):

In [ ]:
from zoneinfo import ZoneInfoNotFoundError, available_timezones

print(ZoneInfo("Europe/London") is london)         # True — cached
print(london.key)                                  # Europe/London

try:
    ZoneInfo("Mars/Olympus_Mons")
except ZoneInfoNotFoundError as e:
    print(f"{type(e).__name__}: {e}")

print(len(available_timezones()) > 400, sorted(available_timezones())[:3])

⚠️ **Gotcha:** use IANA keys (`"America/New_York"`), not Windows ids (`"Eastern Standard Time"`) and not abbreviations (`"EST"`, `"BST"`) — abbreviations are ambiguous and are not zone keys.

## Where does the time zone data come from? (`tzdata`)

`zoneinfo` contains the *code*; the *data* comes from the operating system (`/usr/share/zoneinfo` on Linux and macOS). Windows has no IANA database, so there `zoneinfo` falls back to the first-party **`tzdata`** package from PyPI (`uv add tzdata`). It is a good idea to depend on `tzdata` anyway if your code runs on Windows or in slim containers.

In [ ]:
import zoneinfo

print("OS directories searched:", len(zoneinfo.TZPATH))   # machine-specific; often 0 on Windows
try:
    import tzdata  # noqa: F401
    print("tzdata package available as a fallback")
except ImportError:
    print("tzdata not installed — relying on the operating system database")

## Converting between zones: `astimezone`

`astimezone(tz)` converts an aware datetime to another zone. The **instant stays the same**, only the wall-clock reading changes — just like `TimeZoneInfo.ConvertTime`.

```csharp
// C#
var meeting = new DateTimeOffset(2026, 7, 1, 9, 0, 0, TimeSpan.FromHours(1));
var tokyo = TimeZoneInfo.ConvertTime(meeting, TimeZoneInfo.FindSystemTimeZoneById("Asia/Tokyo"));
```

In [ ]:
meeting = datetime(2026, 7, 1, 9, 0, tzinfo=ZoneInfo("Europe/London"))

for key in ["UTC", "America/New_York", "Asia/Tokyo", "Australia/Sydney"]:
    local = meeting.astimezone(ZoneInfo(key))
    print(f"{key:18} {local:%Y-%m-%d %H:%M %Z (%z)}")

Comparison and subtraction of aware datetimes use the **instant**, so the same moment in two zones is equal:

In [ ]:
tokyo = meeting.astimezone(ZoneInfo("Asia/Tokyo"))
print(tokyo == meeting)                        # True — same instant
print(tokyo - meeting)                         # 0:00:00

new_york_9am = datetime(2026, 7, 1, 9, 0, tzinfo=ZoneInfo("America/New_York"))
print(new_york_9am - meeting)                  # 5:00:00 — 9am in New York is 5 hours later

### Attaching vs converting

Two operations look similar but mean very different things:

- `naive.replace(tzinfo=tz)` — **attach** a zone: "this clock reading was taken in `tz`" (C#: `DateTime.SpecifyKind`, or `new DateTimeOffset(dt, offset)`).
- `aware.astimezone(tz)` — **convert**: "show me this instant in `tz`".

In [ ]:
reading = datetime(2026, 7, 1, 9, 0)                        # naive, e.g. from a CSV file

attached = reading.replace(tzinfo=ZoneInfo("America/New_York"))
print(attached)                                             # 09:00 in New York
print(attached.astimezone(UTC))                             # 13:00 UTC

converted = attached.astimezone(ZoneInfo("Europe/London"))
print(converted)                                            # 14:00 in London — same instant

⚠️ **Gotcha:** calling `astimezone()` on a **naive** datetime does not raise — it assumes the naive value is in the machine's *local* time zone. Calling `astimezone()` with no argument converts to the local zone. Both make your results depend on where the code runs, so be explicit.

## Daylight saving transitions

In 2026, UK clocks go **forward** at 01:00 on 29 March (01:00–02:00 never happens) and **back** at 02:00 on 25 October (01:00–02:00 happens twice). Watch the offset change as we step through the autumn transition in UTC:

In [ ]:
start = datetime(2026, 10, 25, 0, 0, tzinfo=UTC)

for i in range(4):
    instant = start + timedelta(minutes=30 * i)
    local = instant.astimezone(london)
    print(f"{instant:%H:%M} UTC → {local:%H:%M %Z}  fold={local.fold}")

### Ambiguous times and `fold`

01:30 on 25 October happens twice. Python disambiguates with the `fold` attribute (PEP 495): `fold=0` is the **first** occurrence (still BST), `fold=1` the **second** (already GMT). .NET would make you call `TimeZoneInfo.IsAmbiguousTime` and pick an offset from `GetAmbiguousTimeOffsets`.

In [ ]:
ambiguous = datetime(2026, 10, 25, 1, 30, tzinfo=london)

for fold in (0, 1):
    value = ambiguous.replace(fold=fold)
    print(f"fold={fold}: {value} {value.tzname()} → {value.astimezone(UTC):%H:%M} UTC")

### Non-existent times

01:30 on 29 March never happens. Python does not raise; it quietly uses the offset from *before* the transition. A round trip through UTC reveals that the clock reading was invalid:

In [ ]:
def exists(local: datetime) -> bool:
    """True if this wall-clock time actually occurs in its zone (C#: !IsInvalidTime)."""
    round_trip = local.astimezone(UTC).astimezone(local.tzinfo)
    return round_trip.replace(tzinfo=None) == local.replace(tzinfo=None)

gap = datetime(2026, 3, 29, 1, 30, tzinfo=london)
print(gap, "exists:", exists(gap))                          # False
print(gap.astimezone(UTC).astimezone(london))               # normalised to 02:30 BST
print(exists(datetime(2026, 3, 29, 2, 30, tzinfo=london)))  # True

### ⚠️ Arithmetic in a zone is wall-clock arithmetic

When you add a `timedelta` to a datetime with a `ZoneInfo`, Python adds to the **wall-clock** reading and then recomputes the offset. Subtracting two datetimes that share the same `tzinfo` also ignores the offsets. Across a DST change, "one day later" is therefore *not* 24 elapsed hours:

In [ ]:
before = datetime(2026, 3, 28, 12, 0, tzinfo=london)
after = before + timedelta(days=1)

print(after)                                           # 2026-03-29 12:00:00+01:00 — same wall time
print(after - before)                                  # 1 day, 0:00:00 (wall-clock difference)
print(after.astimezone(UTC) - before.astimezone(UTC))  # 23:00:00 — actually elapsed

**Tip:** for *elapsed time* (timeouts, durations, billing), do the arithmetic in UTC. For *calendar* intent ("same time tomorrow", "every day at 09:00 local"), do it in the local zone. Knowing which one you mean is the whole trick.

## The current time: use `now(timezone.utc)`, not `utcnow()`

`datetime.utcnow()` returns a **naive** datetime holding UTC values — which most of the library then treats as *local* time (for example, `.timestamp()` and `.astimezone()`). It is deprecated since Python 3.12. Use an aware UTC value instead:

```csharp
// C#
var nowUtc = DateTimeOffset.UtcNow;
```

In [ ]:
import warnings

now_utc = datetime.now(timezone.utc)            # aware — C#: DateTimeOffset.UtcNow
print(now_utc.tzinfo, now_utc.utcoffset())      # UTC 0:00:00 (the time itself changes every run)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    legacy = datetime.utcnow()                  # deprecated: naive!
print(legacy.tzinfo)                            # None
print(caught[0].category.__name__)              # DeprecationWarning

The same applies to `datetime.utcfromtimestamp(ts)` — use `datetime.fromtimestamp(ts, tz=timezone.utc)`. To get the current time in a specific zone, pass the zone: `datetime.now(ZoneInfo("Asia/Tokyo"))`.

## Best practice: store UTC, convert at the edges

1. Convert incoming values to **aware UTC** as soon as they enter the system.
2. Store and compute in UTC (database columns, logs, message payloads — as ISO 8601 with `+00:00`).
3. Convert to the user's zone only when **displaying**.
4. Exception: for *future* local events ("dentist at 09:00 on 1 July in London"), store the local time **and** the zone key, because DST rules can change before the event happens.

In [ ]:
# 1. input arrives as a local reading plus the user's zone
user_zone = ZoneInfo("America/New_York")
entered = datetime(2026, 11, 2, 18, 45).replace(tzinfo=user_zone)

# 2. store as UTC ISO 8601
stored = entered.astimezone(UTC).isoformat()
print(stored)

# 3. display for a different user
viewer_zone = ZoneInfo("Europe/London")
print(datetime.fromisoformat(stored).astimezone(viewer_zone).strftime("%d %b %Y %H:%M %Z"))

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| UTC "now" | `DateTimeOffset.UtcNow` | `datetime.now(timezone.utc)` |
| UTC constant | `TimeZoneInfo.Utc` | `timezone.utc` / `datetime.UTC` |
| Fixed offset | `new DateTimeOffset(dt, TimeSpan.FromHours(1))` | `dt.replace(tzinfo=timezone(timedelta(hours=1)))` |
| Named zone | `TimeZoneInfo.FindSystemTimeZoneById("Europe/London")` | `ZoneInfo("Europe/London")` |
| Attach a zone | `DateTime.SpecifyKind` / `new DateTimeOffset(...)` | `naive.replace(tzinfo=tz)` |
| Convert | `TimeZoneInfo.ConvertTime(dto, tz)` | `aware.astimezone(tz)` |
| Current offset | `tz.GetUtcOffset(dt)` | `dt.utcoffset()` |
| Is DST? | `tz.IsDaylightSavingTime(dt)` | `bool(dt.dst())` |
| Ambiguous time | `tz.IsAmbiguousTime(dt)` | choose with `dt.replace(fold=0/1)` |
| Unknown zone | `TimeZoneNotFoundException` | `ZoneInfoNotFoundError` |
| Zone data on Windows | built in | `tzdata` package |

## Exercises

1. A flight leaves London at 2026-07-01 10:15 local time and lands in Tokyo at 2026-07-02 07:30 local time. Compute the flight duration.
2. Write `to_utc(text: str, zone_key: str) -> datetime` that parses a naive `"YYYY-MM-DD HH:MM"` string, attaches the given zone and returns an aware UTC datetime. Test it with `"2026-12-24 18:00"` in `"America/Los_Angeles"`.
3. Write `is_ambiguous(local: datetime) -> bool` that returns `True` when a zone-aware wall-clock time occurs twice. Test 2026-10-25 01:30 and 02:30, and 2026-03-29 01:30, in `Europe/London`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from datetime import datetime, UTC
from zoneinfo import ZoneInfo

# 1 — different zones, so subtraction compares instants
departure = datetime(2026, 7, 1, 10, 15, tzinfo=ZoneInfo("Europe/London"))
arrival = datetime(2026, 7, 2, 7, 30, tzinfo=ZoneInfo("Asia/Tokyo"))
print(arrival - departure)          # 13:15:00

# 2 — attach (replace), then convert (astimezone)
def to_utc(text: str, zone_key: str) -> datetime:
    naive = datetime.strptime(text, "%Y-%m-%d %H:%M")
    return naive.replace(tzinfo=ZoneInfo(zone_key)).astimezone(UTC)

print(to_utc("2026-12-24 18:00", "America/Los_Angeles"))   # 2026-12-25 02:00:00+00:00

# 3 — fold=0 and fold=1 give different offsets for ambiguous times...
#     ...but also for non-existent (gap) times, so check the reading survives a UTC round trip
def is_ambiguous(local: datetime) -> bool:
    first, second = local.replace(fold=0), local.replace(fold=1)
    if first.utcoffset() == second.utcoffset():
        return False
    round_trip = first.astimezone(UTC).astimezone(local.tzinfo)
    return round_trip.replace(tzinfo=None) == local.replace(tzinfo=None)

london = ZoneInfo("Europe/London")
print(is_ambiguous(datetime(2026, 10, 25, 1, 30, tzinfo=london)))   # True
print(is_ambiguous(datetime(2026, 10, 25, 2, 30, tzinfo=london)))   # False
print(is_ambiguous(datetime(2026, 3, 29, 1, 30, tzinfo=london)))    # False — a gap, not ambiguous
```

</details>